In [4]:
import sys
path_to_pip_installs = "/tmp/test_env"
if path_to_pip_installs not in sys.path:
    sys.path.insert(0, path_to_pip_installs)

path = "/home/students/studweilc1/MU-Diff/"
# add this to path for imports
if path not in sys.path:
    sys.path.append(path)

import torch
import numpy as np
import os
from backbones.dense_layer import conv2d
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from dataset.dataset_dixon import CreateDatasetSynthesis_with_masks
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio as psnr

from backbones.registration_network import RegistrationUNet

W0930 17:19:19.935000 958409 torch/utils/cpp_extension.py:118] No CUDA runtime is found, using CUDA_HOME='/home/students/studweilc1/.conda/envs/cornelius_new'
W0930 17:19:20.021000 958409 torch/utils/cpp_extension.py:2425] TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
W0930 17:19:20.021000 958409 torch/utils/cpp_extension.py:2425] If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'] to specific architectures.
W0930 17:19:20.041000 958409 torch/utils/cpp_extension.py:2425] TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
W0930 17:19:20.041000 958409 torch/utils/cpp_extension.py:2425] If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'] to specific architectures.


module_path = /home/students/studweilc1/MU-Diff/utils/op


In [ ]:
input_path = "/home/students/studweilc1/MU-Diff/data/my_data2"
batch_size = 1

def dice_loss(pred, target, smooth=1.):
    target = (target + 1) / 2.0
    pred = (pred + 1) / 2.0
    intersection = (pred * target).sum(dim=(2, 3))
    union = (pred+target).sum(dim=(2, 3))
    dice = (2. * intersection + smooth) / (union + smooth)
    return 1-dice.mean()

def create_warp_grid(flow, base_grid_xy):
    """
    Creates a sampling grid for F.grid_sample from a flow field and a base coordinate grid.
    
    Args:
        flow (torch.Tensor): The optical flow field, shape [B, 2, H, W].
        base_grid_xy (torch.Tensor): The base coordinate grid, shape [B, 2, H, W].
        
    Returns:
        torch.Tensor: A grid suitable for F.grid_sample, shape [B, H, W, 2].
    """
    B, _, H, W = flow.shape
    
    # Add flow offsets to the base coordinates
    v_grid = base_grid_xy + flow

    # Scale grid to [-1, 1] for grid_sample
    v_grid_scaled_x = 2.0 * v_grid[:, 0, :, :] / max(W - 1, 1) - 1.0
    v_grid_scaled_y = 2.0 * v_grid[:, 1, :, :] / max(H - 1, 1) - 1.0
    v_grid_scaled = torch.stack([v_grid_scaled_x, v_grid_scaled_y], dim=1)

    # Permute to [B, H, W, 2] for grid_sample
    v_grid_final = v_grid_scaled.permute(0, 2, 3, 1)
    return v_grid_final



dataset = CreateDatasetSynthesis_with_masks(phase="train", input_path=input_path)
dataset_val = CreateDatasetSynthesis_with_masks(phase="val", input_path=input_path)

data_loader = torch.utils.data.DataLoader(dataset,
                                            batch_size=batch_size,
                                            shuffle=False,
                                            num_workers=4,
                                            pin_memory=False,
                                            drop_last=True)

data_loader_val = torch.utils.data.DataLoader(dataset_val,
                                                batch_size=batch_size,
                                                shuffle=False,
                                                num_workers=4,
                                                pin_memory=False,
                                                drop_last=True)

padding in x-y with:0-0
padding in x-y with:0-0
padding in x-y with:0-0
padding in x-y with:0-0
padding in x-y with:0-0
padding in x-y with:0-0
padding in x-y with:0-0
padding in x-y with:0-0
padding in x-y with:0-0
padding in x-y with:0-0
padding in x-y with:0-0
padding in x-y with:0-0
padding in x-y with:0-0
padding in x-y with:0-0
padding in x-y with:0-0
padding in x-y with:0-0


In [ ]:
device = "cpu"
reg_net = RegistrationUNet().to(device)
reg_net.zero_grad()

for iteration, (x1, x2, x3, x4, m1, m2, m3, m4) in enumerate(data_loader):
images = [d.to(device, non_blocking=True) for d in [x1, x2, x3, x4]]
masks = [d.to(device, non_blocking=True) for d in [m1, m2, m3, m4]]
reg_input = torch.cat([cond_data1, cond_data2, cond_data3, target_mask], dim=1)
flow1, flow2, flow3 = reg_net(reg_input)

# Create warping grids and warp inputs
grid1 = create_warp_grid(flow1, base_grid_xy)
grid2 = create_warp_grid(flow2, base_grid_xy)
grid3 = create_warp_grid(flow3, base_grid_xy)

cond1_warped = F.grid_sample(cond_data1, grid1, mode='bilinear', padding_mode='border', align_corners=True)
cond2_warped = F.grid_sample(cond_data2, grid2, mode='bilinear', padding_mode='border', align_corners=True)
cond3_warped = F.grid_sample(cond_data3, grid3, mode='bilinear', padding_mode='border', align_corners=True)

mask1_warped = F.grid_sample(mask1, grid1, mode='nearest', align_corners=True)
mask2_warped = F.grid_sample(mask2, grid2, mode='nearest', align_corners=True)
mask3_warped = F.grid_sample(mask3, grid3, mode='nearest', align_corners=True)